<a href="https://colab.research.google.com/github/Nebius-Academy/LLM-Engineering-Essentials/blob/main/topic2/2.5_establishing_non_linear_reasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Открыть в Colab"/></a>

# LLM Engineering Essentials от Nebius Academy
Курс на github: [ссылка](https://github.com/Nebius-Academy/LLM-Engineering-Essentials/tree/main)
Курс сейчас находится в разработке, скоро появятся дополнительные материалы. [Подпишитесь, чтобы быть в курсе](https://academy.nebius.com/llm-engineering-essentials/update/)
№ 2.5. Развитие способностей нелинейного рассуждения

В предыдущей тетради мы обсуждали способы организации процесса нелинейного рассуждения. Однако некоторые из сегодняшних LLM (например, **o1**, **o3-mini**, **Gemini** («мыслящий» вид этого семейства), **DeepSeek R1**, **Grok 3** и **Claude 3.7 Sonnet**) способны генерировать нелинейные рассуждения без дополнительного проектирования со стороны пользователя.
В этом блокноте мы обсудим несколько подходов к созданию собственных возможностей нелинейного мышления.
Но прежде чем мы продолжим, давайте еще раз кратко рассмотрим, что такое **нелинейное рассуждение**. В отличие от немедленного нахождения правильного решения, этот подход является исследовательским: экспериментированием, итерациями, самокоррекцией и иногда началом с нуля. По сути, это целое **Древо мыслей**, сжатое в один текст. Его также часто называют **длинным рассуждением** из-за его расширенного следа рассуждения.

# Нелинейные рассуждения и вычисления времени вывода
Как мы обсуждали ранее, одна из причин, по которой цепочка мыслей (CoT) приносит пользу, заключается в том, что она позволяет LLM выполнять больше вычислений «под капотом», а дальнейшее увеличение этих вычислений может принести дополнительную прибыль. В предыдущем блокноте мы расширили вычисления времени вывода за счет оркестрации. **Нелинейное рассуждение** идет еще дальше — по сути, это **родное Древо мыслей**.
Неудивительно, что по мере увеличения длины нелинейных рассуждений (т. е. размера собственного дерева) качество улучшается, но, конечно, за счет возрастающих затрат.
Вот цифры от **OpenAI** о **o1**:
<center>
<img src="https://drive.google.com/uc?export=view&id=1x8gxVEENI8tD2yV2KD1sV8jSuWNXTLDh" width=600 />
[Источник](https://openai.com/index/learning-to-reason-with-llms/)
</center>
А вот цифры из **Anthropic** о **Сонете Клода 3.7**:
<center>
<img src="https://drive.google.com/uc?export=view&id=1cNKRvHupStpMh4EVG4XgwZ9l53N-1E5x" width=600 />
[Источник](https://www.anthropic.com/news/visible-extended-thinking)
</center>
Также неудивительно, что последняя версия **GPT-4.5**, не обладающая возможностями долгого рассуждения, выполняет математические задачи значительно хуже, чем **o3-mini-high**, более ранняя модель, способная к нелинейным рассуждениям.
<img src="https://drive.google.com/uc?export=view&id=1N7hHOFr8SHRbUFWBz5ZQDHxJqnEPVpXW" width=600 />
[Источник](https://openai.com/index/introducing-gpt-4-5/)
</center>

Отличительной особенностью Сонета Клода 3.7 является контролируемая длина рассуждений. Давайте проиллюстрируем это на математической задаче.

In [ ]:
import os

with open("anthropic_api_key", "r") as file:
    nebius_api_key = file.read().strip()

os.environ["ANTHROPIC_API_KEY"] = nebius_api_key

In [ ]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 6.0 MB/s eta 0:00:00


`client.messages.create` от Anthropic имеет дополнительный аргумент `thinking` для **Сонета Клода 3.7**. Если он отключен, вы получите обычный ответ CoT.

In [ ]:
import anthropic

# Problem 22 from the AIME dataset: https://huggingface.co/datasets/AI-MO/aimo-validation-aime
prompt = """Denali and Nate work for a dog walking business and are paid for each dog they walk. Denali is responsible for $16$ dogs and Nate is responsible for $12$ dogs. Under the company's new policy, they will be assigned or unassigned new dogs in groups of $x$ dogs. The ratio of Denali's pay to Nate's pay would be the same if Denali started walking $4x$ more dogs and Nate stayed at $12$ dogs or if $x$ of Nate's dogs were reassigned to Denali. Find $x$ if $x\\neq0$."""


client = anthropic.Anthropic(
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
)
reply = client.messages.create(
    model="claude-3-7-sonnet-20250219",
    max_tokens=1024,
    thinking={
        "type": "disabled",
    },
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(reply.content[0].text)

# Finding $x$ in the Dog Walking Problem

I need to determine the value of $x$ where $x$ represents the number of dogs in a group that can be added or removed from Denali's and Nate's responsibilities.

## Setting up the problem

Given information:
- Denali has 16 dogs
- Nate has 12 dogs
- The ratio of their pay would be the same under two scenarios:
  1. Denali gets 4x more dogs (Nate's stays at 12)
  2. x dogs are transferred from Nate to Denali

## Creating equations

Let me denote Denali's pay rate per dog as $d$ and Nate's pay rate as $n$.

### Initial pay ratio
Initial ratio of Denali's pay to Nate's pay: $\frac{16d}{12n}$

### First scenario
When Denali gets 4x more dogs, the ratio becomes:
$\frac{(16+4x)d}{12n}$

### Second scenario
When x dogs are transferred from Nate to Denali, the ratio becomes:
$\frac{(16+x)d}{(12-x)n}$

### Setting the scenarios equal
Since both scenarios must yield the same ratio:
$\frac{(16+4x)d}{12n} = \frac{(16+x)d}{(12-x)n}$

## Solving the equation


Правильный ответ — $5$, и его можно получить здесь. Давайте также посчитаем количество токенов в ответе. Интерфейс Anthropic для этого немного странный: вам нужно указать любую строку, длину токена которой вы хотите узнать, в качестве сообщения пользователя, и количество токенов будет отображаться как количество входных токенов:

In [ ]:
token_count = client.messages.count_tokens(
    model="claude-3-7-sonnet-20250219",
    messages=[
        {"role": "user", "content": reply.content[0].text}
    ]
)
print(token_count.model_dump_json())

{"input_tokens":568}


Чтобы включить длинные рассуждения, мы устанавливаем для `type` из `thinking` значение `enabled` и устанавливаем некоторый бюджет. Чем больше этот бюджет, тем длиннее (по крайней мере теоретически) будут итоговые рассуждения.

In [ ]:
answers = []

budgets = [1024, 2048, 4096, 8192]

for budget in budgets:
    reply = client.messages.create(
        model="claude-3-7-sonnet-20250219",
        max_tokens=512+budget,
        thinking={
            "type": "enabled",
            "budget_tokens": budget
        },
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    answers.append(reply)

    # Checking actual answer length:
    thinking_token_count = client.messages.count_tokens(
        model="claude-3-7-sonnet-20250219",
        messages=[
            {"role": "user", "content": reply.content[0].thinking}
        ]
    )

    print(f"""For budget={budget}:
          Thinking tokens: {thinking_token_count.model_dump_json()}""")

    if len(reply.content) > 1:
        answer_token_count = client.messages.count_tokens(
            model="claude-3-7-sonnet-20250219",
            messages=[
                {"role": "user", "content": reply.content[1].text}
            ]
        )
        print(f"""          Answer tokens: {answer_token_count.model_dump_json()}""")
    else:
        print("    Reasoning was cut short, no answer generated")

For budget=1024:
          Thinking tokens: {"input_tokens":1035}
          Answer tokens: {"input_tokens":440}
For budget=2048:
          Thinking tokens: {"input_tokens":1120}
          Answer tokens: {"input_tokens":600}
For budget=4096:
          Thinking tokens: {"input_tokens":2159}
          Answer tokens: {"input_tokens":547}
For budget=8192:
          Thinking tokens: {"input_tokens":2102}
          Answer tokens: {"input_tokens":614}


Как видите, количество рассуждений действительно растет (конкретные числа могут различаться), а длина ответа особо не меняется. Теперь, если вы проверите рассуждения, вы увидите, что при больших бюджетах Клод выполняет несколько раундов двойной проверки:

In [ ]:
for budget, answer in zip(budgets, answers):
    print(f"===BUDGET {budget}===")
    print("====REASONING====")
    print(answer.content[0].thinking)
    if len(answer.content) > 1:
        print("====ANSWER====")
        print(answer.content[1].text)
    else:
        print("====NO ANSWER GENERATED====")
    print("\n\n")

===BUDGET 1024===
====REASONING====
Let's think this through step by step.

Initially, Denali has 16 dogs and Nate has 12 dogs. Let's say the pay is proportional to the number of dogs, with proportionality constant $P$.

Denali's initial pay is $16P$.
Nate's initial pay is $12P$.
The ratio of Denali's pay to Nate's pay is $\frac{16P}{12P} = \frac{16}{12} = \frac{4}{3}$.

Scenario 1: Denali gets 4x more dogs, and Nate stays at 12 dogs.
Denali would have $16 + 4x$ dogs, with pay $(16 + 4x)P$.
Nate would still have 12 dogs, with pay $12P$.
The new ratio would be $\frac{(16 + 4x)P}{12P} = \frac{16 + 4x}{12}$.

Scenario 2: x of Nate's dogs are reassigned to Denali.
Denali would have $16 + x$ dogs, with pay $(16 + x)P$.
Nate would have $12 - x$ dogs, with pay $(12 - x)P$.
The new ratio would be $\frac{(16 + x)P}{(12 - x)P} = \frac{16 + x}{12 - x}$.

Since these two ratios are equal, we have:
$\frac{16 + 4x}{12} = \frac{16 + x}{12 - x}$

Let's solve for $x$.
$(16 + 4x)(12 - x) = (16 + x)(12)$

У нас мало что изменилось, чтобы заглянуть в процесс обучения Клода, но, к счастью, мы можем получить некоторую информацию из статьи [s1: Простое масштабирование времени тестирования]([[PH_00002]]]. Мы еще вернемся к ней при обсуждении качества и количества обучающих данных, а прямо сейчас мы обсудим тактику **принуждения к бюджету**, предложенную в этой статье.
Формирование бюджета – удивительно простая идея. Авторы предлагают сделать LLM более длительным, просто вводя `Wait` каждый раз, когда он собирается дать ответ, пока не будет достигнута определенная целевая длина. Как мы уже видели, `Wait` побуждает LLM все перепроверить или, возможно, изучить новый подход.
И действительно, этот подход к увеличению бюджета времени вывода оказывается более эффективным, чем самосогласованность:
<center>
<img src="https://drive.google.com/uc?export=view&id=1lD-NBNW242onaE4_v-qYq3nMGoDVxCNS" width=600 />
</center>

# **Отправная точка для обучения модели нелинейному мышлению**
Во всех случаях, когда подробности обучения общеизвестны, процесс начинается с **предварительно обученной модели**. Например, **DeepSeek R1** обучался, начиная с **DeepSeek V3**.
В этой тетради мы рассмотрим удивительно простые подходы к развитию способностей к нелинейному мышлению. Я считаю, что в основе этих методов лежит ключевой принцип, изложенный в статье [Демистификация рассуждений с длинной цепочкой мыслей в LLM](https://arxiv.org/pdf/2502.03373):
**Основные способности к рассуждению, такие как исправление ошибок, изначально присутствуют в базовых (предварительно обученных) моделях, даже если их активировать может быть сложно.**
Учтите следующее: предварительно обученная модель столкнулась с огромным количеством текста, включая примеры нелинейных рассуждений. Однако, поскольку таких примеров, вероятно, мало, модель обычно не генерирует их самостоятельно. Чтобы скорректировать вероятность использования этого «образа мышления», требуется некоторая изобретательность. В следующих разделах мы более подробно обсудим некоторые из подходов.

# **Классический подход: SFT**
Самый простой способ точной настройки LLM для любой задачи, включая нелинейное рассуждение, — это **контролируемая точная настройка (SFT)**, то есть точная настройка набора данных, состоящего из пар **(промпт, завершение)**, где завершения проявляют желаемые свойства.
Если мы выберем SFT, мы должны сначала собрать набор данных примеров нелинейных рассуждений, что далеко не просто:
- **Наем аннотаторов** обходится дорого, и даже исследователю-математику сложно придумать качественные примеры нелинейных рассуждений. Хороший пример должен исследовать множество путей мышления (в том числе в конечном итоге бесполезных), включать ошибки и демонстрировать самокоррекцию — то, что редко приходит вам в голову.
- **Учебники по математике редко содержат настоящие нелинейные рассуждения.** Доказательства теорем и решения задач обычно представлены в окончательной форме, без особого понимания процесса творческого решения задач.
Однако сравнение **нелинейного рассуждения** с **Древом мыслей** предполагает многообещающий подход для **автоматического создания** таких примеров: мы можем просто взять Древо мыслей и переписать его как единую последовательность!
Подобный подход (поиск по дереву) -> (нелинейное рассуждение) был исследован, например, в статье [*Towards System 2 Reasoning in LLM: Learning How to Think With Meta Chain-of-Thought*]([[PH_00000]]]. Однако вместо простого поиска в ширину/глубину они использовали более сложную стратегию поиска по дереву, основанную на **Поиске по дереву Монте-Карло (MCTS)**. Давайте кратко обсудим что это такое.

## **Поиск в дереве Монте-Карло (MCTS)**
MCTS — это **алгоритм поиска по дереву**, который появился задолго до LLM и широко использовался в игровом искусственном интеллекте, включая знаменитую систему [AlphaGo](https://en.wikipedia.org/wiki/AlphaGo) .
Если мы применим MCTS для оркестровки рассуждений, каждый **узел** в дереве поиска представляет собой **частичное решение**, и каждый узел хранит:
- **$n_i$**: количество посещений этого узла.
- **$v_i$**: совокупное **награда** за все посещения этого узла.
- **Возможные действия** (т. е. возможные следующие шаги рассуждения). В каждый конкретный момент одни действия **опробованы**, а другие остаются **неопробованными**.
Дерево растет шаг за шагом, проходя четыре итерационных этапа: **Выбор, Расширение, Моделирование и Обратное распространение**.
### **1. Выбор**
Здесь мы выбираем **частичное решение** (=узел) для расширения. Начиная с корня (пустого решения), следуем таким правилам:
— Если у **текущего узла есть невыполненные действия**, мы выбираем его.
- В противном случае мы выбираем **дочерний узел**, который максимизирует показатель **верхней границы достоверности (UCB)**:
  $$UCB(i) = \frac{v_i}{n_i} + c\sqrt{\frac{\log{n_p}}{n_i}},$$
  где **$p$** — индекс родительского узла, а **$i$** — индекс его дочернего узла.
  - **первый срок** поощряет **эксплуатацию** (выбор узлов с высокими наградами).
  – **Второй срок** поощряет **исследования** (пробование менее посещаемых узлов).
Кроме того, полезно исключить узлы, все поддерево которых уже полностью исследовано и достигло конечных состояний (т. е. окончательных решений).
### **2. Расширение**
Для выбранного узла мы **исследуем одно не опробованное действие** — добавление нового шага рассуждения к частичному решению.
- Если это **новое частичное решение** **не окончательное** (т. е. еще не содержит окончательного ответа), мы генерируем возможные следующие шаги.
- **Количество сгенерированных следующих шагов** контролируется гиперпараметром (`branching_factor`).
— Ограничение `max_height` предотвращает расширение дерева слишком далеко от корня.
### **3. Моделирование**
Мы оцениваем вновь созданное **частичное решение**.
– Если **Модель вознаграждения процесса** недоступна, мы оцениваем качество с помощью **развертывания** – генерации нескольких завершений и расчета соотношения правильных ответов. Этот подход был бы невозможен при выводе, когда ответы были бы неизвестны, но он хорошо работает для создания набора данных нелинейных рассуждений из доступного `(problem, answer dataset)`.
### **4. Обратное распространение**
Вычисленный балл распространяется вверх по дереву:
- Для **нового частичного решения** и всех его **предшествующих узлов** мы обновляем:
  - $v_j$, добавление вычисленного балла.
  - $n_j$, увеличивая количество посещений на $1$.
Эти четыре этапа **повторяются для `n_iterations`**, после чего мы выбираем **терминальное решение с наивысшим баллом** - или,если такового не существует, мы скорбим и выбираем пень, набравший наибольшее количество очков.
---
Вот простая визуализация:
<center>
<img src="https://drive.google.com/uc?export=view&id=1m3UxM4s_x-kAzprHUh30TQZL7rkORv3O" width=600 />
</center>
Здесь каждый узел показывает
```
v: <mean_value>
n: <n_visits>
```
Конечные узлы изображаются маленькими квадратами, а нетерминальные — кружками.
---
См. практическую часть кода этой визуализации, а также реализацию MCTS для оркестровки LLM.
Интересная особенность MCTS заключается в том, что, как отмечено в [К рассуждению по системе 2 в LLM] (https://arxiv.org/pdf/2501.04682), фактическое рассуждение **o1** и подобных моделей во многом похоже на траектории поиска MCTS.
**Примечание**. Мы могли бы включить MCTS в предыдущий блокнот среди других методов резонансной оркестровки. И действительно, иногда его можно использовать таким образом. Но, тем не менее, его сложность делает MCTS маловероятным инструментом в большинстве ситуаций.

## Качество и количество данных
Может показаться, что обучение LLM нелинейному мышлению потребует огромного количества данных, но оказывается, что качество данных может компенсировать недостаток их количества. Давайте кратко обсудим два примера.
#### [S1: Простое масштабирование времени тестирования](https://arxiv.org/pdf/2501.19393)
Мы уже обсуждали **принуждение к бюджету**, предложенное в этой статье. Теперь пришло время упомянуть, что они доработали свою базовую модель **Qwen2.5-.
32B-Инструктируйте** только по **1000** высококачественным примерам рассуждений.
Как они собирали эти данные:
* Они получили 59 029 вопросов из 16 различных источников (включая множество олимпиадных задач).
* Для каждого вопроса они создали логику рассуждений и решение, используя экспериментальный API Google Gemini Flash Thinking API. Такой подход к сбору данных известен как **дистилляция**, поскольку возможности более крупной/холодной модели перерабатываются в меньшую/более скромную модель.
* Затем они фильтруют проблемы, сначала по *качеству* (отсутствие плохого форматирования и т. д.), затем по *сложности*. Задача считается сложной, если ни **Qwen2.5-7B-Instruct**, ни **Qwen2.5-32B-Instruct** не могут ее решить, а также длина рассуждений достаточно велика.
* Наконец, 1000 задач были стратифицированы по ряду тем.
По соотношению качество+стоимость обучения эта модель очень крутая:
<center>
<img src="https://drive.google.com/uc?export=view&id=1edwrnzsCzSfskwJUgecIcsnO7QOqBatO" width=400 />
</center>
#### [ЛИМО: меньше значит больше для рассуждений](https://arxiv.org/pdf/2502.03387)
Авторы этой статьи берут **NuminaMath** в качестве базовой модели и настраивают ее только на **817** высококачественных тщательно подобранных обучающих выборках, чтобы получить LLM, который обеспечивает весьма впечатляющие математические результаты с исключительным обобщением вне распределения. Просто посмотрите на результаты тестов ниже. (И действительно нужна смелость, чтобы сравнивать себя с **o1**.)
<center>
<img src="https://drive.google.com/uc?export=view&id=1uAj-Dara6G985eLxHT1mVgnIB_T-uxMS" width=600 />
</center>
Также интересно сравнить их модель с базовой **NuminaMath** LLM, что уже довольно круто.
<center>
<img src="https://drive.google.com/uc?export=view&id=1IlomIGaROqDfoT4BDxxuTzlBNuXkzp8q" width=600 />
</center>
Итак, какие рекомендации предлагают авторы, когда дело доходит до высококачественных данных нелинейных рассуждений? Их три:
* **Структурированная организация**. Токены распределяются по отдельным «мыслям» в соответствии с их важностью и сложностью, при этом больше токенов приходится на ключевые аргументы, при этом более простые шаги остаются краткими.
* ***Когнитивные леса**. Концепции вводятся стратегически, с тщательным устранением пробелов, чтобы сделать сложные рассуждения более доступными.
* **Тщательная проверка**. Промежуточные результаты и предположения часто проверяются; логическая последовательность обеспечена.
Что касается третьего пункта, то проверка особенно важна из-за риска появления галлюцинаций. Интересным примером является статья [rStar-Math](https://arxiv.org/pdf/2501.04519), где авторы тренируют свои LLM для создания решений в виде кода Python с текстом в виде комментариев к коду.
<center>
<img src="https://drive.google.com/uc?export=view&id=1Psnpfn5iCdPLKGlj6-GfwnmPt2qU6iVn" width=800 />
</center>
Благодаря агентным возможностям выполнение этого кода может предоставить LLM ценную обратную связь, что в конечном итоге улучшит его рассуждения.
**ЛИМО** — это, конечно, только один пример. Например, авторы DeepSeek R1 использовали тысячи примеров на этапе тонкой настройки обучения с холодным запуском.

# Новый подход: обучение с подкреплением (RL).
Прежде чем углубиться в рассуждения, давайте кратко обсудим, почему традиционной контролируемой точной настройки (SFT) иногда может быть недостаточно.
SFT хорошо работает, когда у нас есть набор данных из пар `(prompt, answer)` — по сути, когда мы можем четко указать, что должна генерировать модель, чтобы она соответствовала нашим ожиданиям. Однако что, если мы хотим научить LLM создавать **нетоксичный** текст? Простого предоставления примеров нетоксичных результатов недостаточно, чтобы научить модель тому, что переходит черту токсичности. Без четких указаний о том, что неприемлемо, модель не узнает границ, которые ей не следует пересекать.
С другой стороны, обучить **модель вознаграждения**, чтобы отличать токсичный текст от нетоксичного, относительно просто. Итак, **вместо того, чтобы явно показывать LLM, что генерировать, мы обучаем его максимизировать сигнал вознаграждения.** И именно здесь в игру вступает **Обучение с подкреплением (RL)**!

##RL в двух словах (можете пропустить, если вы не новичок в теме)
Представьте, что вы хотите обучить ИИ-бота играть в [Принца Персии](https://www.youtube.com/watch?v=FGQmtlxllWY) (игра 1989 года). В этой игре персонаж игрока (то есть титульный принц) может:
* Идите влево или вправо, прыгайте и сражайтесь со стражниками своим мечом
* Падайте в ямы, натыкайтесь на шипы или убивайте охранники.
* Не хватит времени и проиграйте
* Спасите принцессу и победите
<center>
<img src="https://drive.google.com/uc?export=view&id=1jye3OWWmdRr2dWSDQw5NUSFa_jOyJfJK" width=600 />
</center>
Простейшим ботом с искусственным интеллектом была бы нейронная сеть, которая принимает текущий экран (или, может быть, несколько недавних экранов) в качестве входных данных и прогнозирует следующее действие — но как ее обучить?
Парадигма обучения с учителем, вероятно, потребует от нас сыграть во множество успешных игр, записать все экраны и обучить модель прогнозировать выбранные нами действия. Но у этого подхода есть несколько проблем, в том числе следующие:
* Игра довольно длинная, поэтому собирать достаточное количество раундов было бы слишком утомительно.
* Недостаточно показать правильные способы игры; бот также должен изучить движения, которых следует избегать.
* В игре предусмотрено огромное количество возможных действий на самых разных экранах. Разумно ожидать, что успешные игры, в которые играют опытные игроки, не будут генерировать данные с уровнем необходимого разнообразия, чтобы бот «понял» все распределение действий.
Итак, эти соображения заставляют нас рассмотреть возможность обучения бота методом **проб и ошибок**:
1. Каким-то образом инициализировать его поведение («**policy**»).
2. Разрешить ему играть по этой политике, проверять различные ходы (в том числе очень неловкие) и получать удовольствие от падения на дно ямы и т.д.
3. Корректировать политику в зависимости от ее успеха или неудач.
4. Повторяем шаги 2 и 3, пока нам не надоест ждать или пока бот не научится играть в Prince of Persia как профессионал.
Давайте немного формализуем это, используя общепринятую терминологию RL:
* (Наблюдаемое) **состояние** — это информация, которой мы располагаем об игре в настоящий момент. В нашем случае это содержимое текущего экрана.
* **Агент** — это бот, способный выполнять несколько **действий**.
* **Окружающая среда** — это игра. Он определяет возможные состояния, возможные действия и влияние каждого действия на текущее состояние, а также то, какое состояние будет следующим.
* **Политика** — это (обучаемая) стратегия, которую бот использует в игре. В нашем случае это нейронная сеть, которая прогнозирует действия с учетом текущего состояния или истории состояния.
* **Награда** — это баллы, которые мы присваиваем штатам. Например, победа над охранником, переход на следующий уровень или победа в игре могут иметь положительные награды, тогда как падение в яму или ранение охранником будет означать отрицательные награды.
<center>
<img src="https://drive.google.com/uc?export=view&id=1PqJaNX6bvSPHIc6JCNStIo7v7YKzTKHM" width=600 />
</center>
Цель оОбучение – это поиск политики, которая максимизирует вознаграждение, и есть много способов добиться этого.
Теперь мы увидим, что обучение LLM иногда имеет большое значение для Prince of Persia.

## RL в обучении LLM
Вот самый простой способ проявления RL в обучении LLM:
* **Агент** – это наш LLM.
* Наблюдаемое **состояние** является промптом.
* **Действие** — создание завершения.
* **Награда** — это оценка **модели вознаграждения**.
<center>
<img src="https://drive.google.com/uc?export=view&id=1sk9h95G_JsU7QkZiDAcCeRZmfP-tlVrZ" width=600 />
</center>
Эта схема намного проще: всего одна итерация состояния -> действия -> вознаграждения.
Вероятно, наиболее влиятельным методом обучения, основанным на RL, является **RLHF** (что означает просто **Обучение с подкреплением с обратной связью с человеком**), который до сих пор используется для установления соответствия LLM человеческим ценностям и предпочтениям - полезности, безвредности и т. д., включая ненавязчивость. (Хотя сейчас его часто заменяют альтернативами без RL, такими как прямая оптимизация политики)

##RL для рассуждений: как DeepSeek обучал R1-Zero
Вы, вероятно, задаетесь вопросом, что общего у всего этого с рассуждением. Совершенно революционной вещью, которую совершил DeepSeek, было обучение их модели **R1-Zero** нелинейным рассуждениям не только без набора данных `(prompt, long reasoning)`, но даже без модели вознаграждения процесса. Для этого они взяли существующую модель **DeepSeek V3** и обучили ее с помощью **Обучения с подкреплением** для оптимизации вознаграждений двух видов:
* **Награда за точность** — это просто точность ответа. Так что нет, решение должно привести только к правильному ответу, каким бы странным он ни был.
* **Формат после вознаграждения** заставляет модель заключать свой мыслительный процесс между `<think>` и `</think>`, как мы видели в примерах.
И было ужасно интересно видеть, что, несмотря на отсутствие специальных указаний по этому поводу, R1Zero не только научился выдавать более длинные и длинные решения.
<center>
<img src="https://drive.google.com/uc?export=view&id=1lO0wN7fbsgLSsrWojHh68YgF-FVXNTEi" width=600 />
[Источник](https://arxiv.org/pdf/2501.12948)
</center>
но также начал демонстрировать человеческие модели выражения мыслей:
<center>
<img src="https://drive.google.com/uc?export=view&id=14z0RVH_GyHaoRXzKasnM5QMX4z_PLolW" width=600 />
[Знаменитый момент ага](https://arxiv.org/pdf/2501.12948)
</center>

Итак, почему это сработало? Чтобы ответить, давайте вернемся к одному из основных моментов этой темы: **хорошо предварительно обученный LLM уже обладает скрытыми способностями к рассуждению, и нам нужно только пробудить их**. **DeepSeek V3** уже был вполне работоспособной моделью и определенно «видел» некоторое количество нелинейных рассуждений во время предварительного обучения. RL помогло превратить эти скрытые знания в активные навыки.
И это не предположение. Авторы книги [Понимание R1-Zero-Like Training: A Critical Perspective](https://arxiv.org/pdf/2503.20783) действительно обнаружили моменты ага в результатах базовой модели **DeepSeek V3**.
<center>
<img src="https://drive.google.com/uc?export=view&id=11UGI55mS6hwzxPK0saPSudAw3RCPMutE" width=500 />
[Источник](https://arxiv.org/pdf/2503.20783)
</center>
Итак, повторюсь, RL не создало новых возможностей, а, скорее, усилило их.
Еще один интересный эксперимент был описан в статье [Четыре навыка высокоэффективных STaRs](https://arxiv.org/pdf/2503.01307). Авторы тренировали как **Qwen-2.5-3B**, так и **Llama-3.2-3B** длительные рассуждения с RL и обнаружили, что Квен тренировалась довольно хорошо, в то время как Лама не показывала большого прогресса.
<center>
<img src="https://drive.google.com/uc?export=view&id=1JNOY7klF1DKWSwn77ZMpUAep2Zfjis1V" width=800 />
[Источник](https://arxiv.org/pdf/2503.20783)
</center>
Чтобы понять это, авторы рассмотрели четыре шаблона длинных рассуждений — проверку, постановку подцелей, возврат и обратную цепочку — и проверили, в какой степени они присутствуют в базовых моделях Квен и Ламы. Квен систематически лучше справлялась со следующими паттернами:
<center>
<img src="https://drive.google.com/uc?export=view&id=1H-4Obb4SRo-GmtTzcWC_qEot199WvZv9" width=400 />
[Источник](https://arxiv.org/pdf/2503.20783)
</center>
Объясняет ли эта разница расхождение в обучении RL? Ну, возможно. Чтобы проверить это, авторы взяли корпус OpenWebMath и создали на его основе два набора данных SFT — один с примерами, демонстрирующими четыре длинных рассуждения, а другой — с примерами, лишенными этих шаблонов. Затем они настроили Llama на каждом из них, прежде чем подвергнуть его RL решению довольно игрушечной, но связанной с математикой задачи.
Результаты оказались весьма интересными: после SFT на примерах, демонстрирующих четыре длинных шаблона рассуждения, Лама реагировала на RL гораздо лучше, чем после SFT на другом наборе данных:
<center>
<img src="https://drive.google.com/uc?export=view&id=134Wy-AE4P0MTGq6lZL2B05LW5JIyN_ET" width=400 />
[Источник](https://arxiv.org/pdf/2503.20783)
</center>
Итак, еще раз повторю: никакое количество RL без начальных возможностей вряд ли поможет!

**Краткий обзор технических особенностей RL**
Мы не будем углубляться в конкретный алгоритм RL, используемый DeepSeek. Это не только потому, что мы не хотим отпугивать вас математикой. Кажется, что здесь можно использовать разные алгоритмы, если только авторам повезет и они будут достаточно способны заставить их работать, что совсем не просто — RL, как известно, сложен. Мы лишь кратко упомянем два из них:
* DeepSeek использовал недавно разработанный ими алгоритм под названием **GPRO** (**оптимизация групповой относительной политики**). Его интересная особенность заключается в том, что награда или, точнее, **преимущество** оценивается из мини-пакета:
  * Для приглашения $q$ генерируется несколько ответов $y_1,\ldots,y_G$.
  * За каждый $y_i$ рассчитывается вознаграждение $r_i$.
  * **Относительное преимущество** рассчитывается для каждого ответа как
  $$A_i = \frac{r_i - \mathrm{mean}(r_1,\ldots,r_G)}{\mathrm{std}(r_1,\ldots,r_G)}$$
  Это помогает стабилизировать тренировку.
* Проект [Open Reasoning Zero](https://github.com/Open-Reasoner-Zero/Open-Reasoner-Zero) просто использует традиционную PPO (оптимизацию проксимальной политики), знакомую по RLHF – но, что удивительно, даже без штрафа KL :O
**Примечание**. Авторы DeepSeek действительно пытались использовать модели процессного вознаграждения, но это не сработало. Это помогает подчеркнуть, что PRM — существа капризные.

##RL недостаточно: от R1-Zero до R1
Несмотря на наличие возможностей нелинейного рассуждения, **DeepSeek R1-Zero** имеет такие проблемы, как плохая читаемость и смешение языков. Так, в обучение финального **DeepSeek-R1** было внесено несколько улучшений. Вот как выглядел его процесс обучения:
* Во-первых, точная настройка нескольких тысяч высококачественных данных, созданных с помощью CoT и уточненных аннотаторами-людьми. (И вполне возможно, что это получено из уже существующих LLM с длительными рассуждениями.) Это было сделано для улучшения нестабильного холодного запуска RL +, чтобы дать модели некоторые начальные рекомендации относительно того, чему следует научиться во время RL. Это звучит как хорошая идея.
* Затем РЛ.
* Затем выполните дополнительную SFT как для рассуждений, так и для нерассуждений (таких как письмо, фактический контроль качества, самопознание и перевод). Частично на этом этапе пытаются решить известные после RL проблемы с читаемостью.
* Наконец, еще один этап RL для установления соответствия человеческим предпочтениям. (Аналог RLHF.)

# Выводы
В этих трех блокнотах мы рассмотрели две самые актуальные темы, связанные с LLM: **Рассуждение LLM** и **Вычисление во время вывода**. Теперь вы знаете:
* Когда полагаться на рассуждения, а когда сдерживаться.
* Как распределить бюджет времени на выводы и выбрать один из различных вариантов.
* Что такое нелинейное рассуждение, как оно устанавливается и почему оно вдруг стало таким популярным.
Мы еще не затрагивали тему, тесно связанную с рассуждениями — планирование и выполнение LLM — но мы обсудим это, когда представим агентов LLM. Следите за обновлениями курса!

# Практическое занятие

## Готовимся

In [ ]:
!pip install -q openai

In [ ]:
import os

with open("nebius_api_key", "r") as file:
    nebius_api_key = file.read().strip()

os.environ["NEBIUS_API_KEY"] = nebius_api_key

Ниже приводится отредактированный текст с пояснением внесенных изменений:
---
## Практика, часть 1: MCTS
В этом разделе мы поделимся нашей реализацией поиска по дереву Монте-Карло (MCTS) и выделим некоторые из его ключевых функций:
- **Параллельное внедрение:**
  Поскольку мы используем развертывание для оценки частичных решений, наш подход включает в себя множество звонков в LLM. Это дорого — поэтому следите за использованием API — и, если делать это последовательно, может быть очень медленно. Поэтому выгодно распараллелить эти вызовы. К сожалению, итерации MCTS по своей сути являются последовательными, поэтому единственная часть, которую можно распараллелить, — это развертывание. Мы достигаем этого, используя `client.chat.completions.create(..., n=number_of_rollouts)`.
- Вы можете настроить несколько параметров:
  - `max_depth`: максимальная глубина дерева поиска.
  - `num_rollouts`: количество полных продолжений, созданных для оценки частичного решения.
  - `num_iterations`: количество выполняемых циклов выбора-расширения-моделирования-обратного распространения ошибки.
  - `branching_factor`: сколько непроверенных действий выполнить для узла при его расширении.
  - `c`: коэффициент в формуле `ucb = exploitation + self.c * exploration`. Этот параметр помогает сбалансировать равномерное исследование дерева и устойчивое движение к цели. Мы рекомендуем начинать с более низких значений `c`, отдавая предпочтение эксплуатации. Если вы уверены, что найдете ответ за разумное количество итераций, вы можете попробовать расширить исследование.
- Сгенерированный ответ сравнивается с фактическим ответом с помощью функции оценки, передаваемой в MCTS, которая по умолчанию равна `answer_comparison_evaluator(generated_answer, true_answer)`. Если ваш ответ представляет собой формулу LaTeX и вам необходимо вычислить ее значение перед сравнением, вы можете переопределить этот оценщик.
Мы рекомендуем вам поэкспериментировать с MCTS для решения различных задач. Просто помните, что он значительно дороже, чем Beam Search, из-за дополнительных затрат на развертывание.

In [ ]:
import numpy as np
import random
from typing import List, Dict, Optional, Tuple, Callable, Any
from dataclasses import asdict, dataclass
import json
import re

@dataclass
class NodeStats:
    """Statistics for a node in the MCTS tree"""
    visits: int
    value: float
    success_rate: float
    depth: int
    is_terminal: bool
    num_children: int

    def to_dict(self) -> Dict:
        return asdict(self)

@dataclass
class MCTSNode:
    state: str  # Current solution state
    parent: Optional['MCTSNode']
    children: List['MCTSNode']
    visits: int
    value: float
    untried_actions: List[str]  # Possible next steps
    is_terminal: bool
    correct_continuations: int = 0
    total_continuations: int = 0

    def get_stats(self) -> NodeStats:
        """Get statistics for this node"""
        return NodeStats(
            visits=self.visits,
            value=self.value,
            success_rate=self.value / max(self.visits, 1),
            depth=self.get_depth(),
            is_terminal=self.is_terminal,
            num_children=len(self.children)
        )

    def get_depth(self) -> int:
        """Get the depth of this node in the tree"""
        depth = 0
        node = self
        while node.parent:
            depth += 1
            node = node.parent
        return depth

    def to_dict(self) -> Dict:
        """Convert node to dictionary representation"""
        return {
            'state': self.state,
            'stats': self.get_stats().to_dict(),
            'children': [child.to_dict() for child in self.children]
        }

class LLMClient:
    """Wrapper for OpenAI-compatible API clients."""
    def __init__(
        self,
        client,
        model: str,
        default_temperature: float = 0.7,
        default_max_tokens: int = 1024,
        system_prompt: Optional[str] = None
    ):
        self.client = client
        self.model = model
        self.default_temperature = default_temperature
        self.default_max_tokens = default_max_tokens
        self.system_prompt = system_prompt

    def generate(
        self,
        prompt: str,
        temperature: Optional[float] = None,
        max_tokens: Optional[int] = None,
        system_prompt: Optional[str] = None,
        n: int = 1
    ) -> List[str]:
        """Generate completions with consistent interface."""
        messages = []

        # Use provided system prompt or fall back to default
        current_system_prompt = system_prompt or self.system_prompt
        if current_system_prompt:
            messages.append({"role": "system", "content": current_system_prompt})

        messages.append({"role": "user", "content": prompt})

        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=temperature if temperature is not None else self.default_temperature,
                max_tokens=max_tokens if max_tokens is not None else self.default_max_tokens,
                n=n
            )
            return [choice.message.content.strip() for choice in response.choices]
        except Exception as e:
            return [""] * n

class MCTS:
    def __init__(self,
                 llm_client: LLMClient,
                 exploration_weight: float = 1.414,
                 num_rollouts: int = 5,
                 max_depth: int = 5,
                 branching_factor: int = 3,
                 c: float = 1.):
        """
        Initialize MCTS for LLM-based problem solving.

        Args:
            llm_client: LLMClient instance
            exploration_weight: UCB1 exploration parameter
            num_rollouts: Number of Monte Carlo rollouts for evaluation
            max_depth: Maximum depth of the search tree
        """
        self.llm_client = llm_client
        self.exploration_weight = exploration_weight
        self.num_rollouts = num_rollouts
        self.max_depth = max_depth
        self.search_tree = None
        self.branching_factor = branching_factor
        self.c = c

    def ucb1(self, node: MCTSNode, parent_visits: int) -> float:
        """Calculate UCB1 value for node selection."""
        if node.visits == 0:
            return float('inf')

        exploitation = node.value / node.visits
        exploration = self.exploration_weight * np.sqrt(np.log(parent_visits) / node.visits)
        return exploitation + self.c * exploration

    def get_possible_actions(self, state: str, problem: str) -> List[str]:
        """Get possible next steps from current state using LLM."""
        prompt = f"""Given the math problem and current partial solution:
        Problem: {problem}
        Current solution steps:
        {state}\n\n"""

        prompt += """Generate the next logical step in the solution. Be concise and direct.
        - Each step should be a single, clear logical or math operation
        - Don't repeat previous steps
        - If during the next step you get the final answer for the problem, you immediately output this answer in \\boxed\{\}. This is very important!

        Next step:"""

        responses = self.llm_client.generate(prompt, n=self.branching_factor)
        return [step for step in responses if step.strip()]

    def evaluate_partial_solution(self,
                               state: str,
                               problem: str,
                               evaluate_continuation: Callable[[str, str], Tuple[int, int]]) -> Tuple[int, int]:
        """
        Evaluate partial solution using the provided evaluation function.
        Returns (correct_continuations, total_continuations)
        """
        return evaluate_continuation(state, problem)

    def select(self, node: MCTSNode) -> MCTSNode:
        """Select most promising node using UCB1."""
        while node.children and not node.is_terminal:
            if node.untried_actions:
                return node

            node = max(node.children,
                      key=lambda n: self.ucb1(n, node.visits))
        return node

    def expand(self,
              node: MCTSNode,
              problem: str,
              evaluate_continuation: Callable[[str, str], Tuple[int, int]]) -> Optional[MCTSNode]:
        """Expand node by adding a child with an untried action."""

        if not node.untried_actions or node.is_terminal:
            return None

        action = random.choice(node.untried_actions)
        node.untried_actions.remove(action)

        new_state = f"{node.state}\n{action}" if node.state else action
        possible_actions = self.get_possible_actions(new_state, problem)

        actual_depth = node.get_depth() + 1  # +1 for the child we're creating
        is_terminal = (actual_depth >= self.max_depth) or ("\\boxed" in new_state)
        correct_cont, total_cont = self.evaluate_partial_solution(
            new_state, problem, evaluate_continuation)

        child = MCTSNode(
            state=new_state,
            parent=node,
            children=[],
            visits=0,
            value=0.0,
            untried_actions=possible_actions,
            is_terminal=is_terminal,
            correct_continuations=correct_cont,
            total_continuations=total_cont
        )

        node.children.append(child)
        return child

    def rollout(self,
            node: MCTSNode,
            problem: str,
            evaluate_continuation: Callable[[str, str], Tuple[int, int]]) -> float:
        """
        Run multiple rollouts by generating full continuations from the node's state.
        For each rollout, generate a complete solution by calling the LLM,
        then evaluate that complete solution using the provided deterministic evaluator.
        Returns an average score (ratio of correct to total continuations).
        """
        # Construct a prompt that asks the LLM to complete the chain-of-thought
        prompt = f"""You are given the math problem and current partial solution:
        Problem: {problem}
        Current solution steps:
        {node.state}\n\n"""

        prompt += """Complete the solution and output the final answer in \\boxed\{\}"""

        # Generate multiple full continuations
        responses = self.llm_client.generate(prompt, n=self.num_rollouts)

        scores = []
        for response in responses:
            # Construct the full solution by appending the generated continuation
            full_solution = f"{node.state}\n{response}"

            # Evaluate the full solution using the deterministic evaluator
            correct, total = evaluate_continuation(full_solution, problem)
            score = correct / max(total, 1)
            scores.append(score)

        # Return the average score over all generated completions
        aggregated_score = sum(scores) / len(scores) if scores else 0.0
        return aggregated_score

    def backpropagate(self, node: MCTSNode, value: float):
        """Backpropagate value up the tree."""
        while node:
            node.visits += 1
            node.value += value
            node = node.parent

    def search(self,
              problem: str,
              evaluate_continuation: Callable[[str, str], Tuple[int, int]],
              initial_state: str = "",
              num_iterations: int = 100,
              verbose: bool = False) -> Tuple[str, MCTSNode]:
        """
        Perform MCTS search to find solution.

        Args:
            problem: Problem description
            evaluate_continuation: Function to evaluate solution continuations
            initial_state: Starting point for solution
            num_iterations: Number of MCTS iterations
            verbose: Whether to print detailed logs

        Returns:
            Best solution found and the search tree root
        """
        possible_actions = self.get_possible_actions(initial_state, problem)
        root = MCTSNode(
            state=initial_state,
            parent=None,
            children=[],
            visits=0,
            value=0.0,
            untried_actions=possible_actions,
            is_terminal=False
        )

        for iteration in range(num_iterations):
            node = self.select(root)
            child = self.expand(node, problem, evaluate_continuation)

            if child:
                # Use multi-rollout simulation to get a robust estimate
                simulation_value = self.rollout(child, problem, evaluate_continuation)
                self.backpropagate(child, simulation_value)

            if verbose and iteration % 5 == 0:
                tree_info = self.get_tree_summary(root)
                print(f"\nIteration {iteration} - Tree Statistics:")
                print(json.dumps(tree_info['statistics'], indent=2))

        # Store the search tree for later analysis
        self.search_tree = root

        best_path = self._get_best_path(root)
        final_solution = best_path[-1]['state']
        return final_solution, root

    def get_tree_summary(self, node: MCTSNode, max_depth: Optional[int] = None) -> Dict:
        """
        Get a summary of the search tree up to max_depth.
        If max_depth is None, returns the entire tree.
        """
        return {
            'tree_structure': node.to_dict(),
            'statistics': {
                'total_nodes': self._count_nodes(node),
                'max_depth': self._get_max_depth(node),
                'leaf_nodes': self._count_leaf_nodes(node),
                'avg_branching': self._get_avg_branching(node),
                'best_path': self._get_best_path(node)
            }
        }

    def _count_nodes(self, node: MCTSNode) -> int:
        """Count total nodes in tree"""
        return 1 + sum(self._count_nodes(child) for child in node.children)

    def _get_max_depth(self, node: MCTSNode) -> int:
        """Get maximum depth of tree"""
        if not node.children:
            return 0
        return 1 + max(self._get_max_depth(child) for child in node.children)

    def _count_leaf_nodes(self, node: MCTSNode) -> int:
        """Count leaf nodes in tree"""
        if not node.children:
            return 1
        return sum(self._count_leaf_nodes(child) for child in node.children)

    def _get_avg_branching(self, node: MCTSNode) -> float:
        """Calculate average branching factor"""
        total_nodes = self._count_nodes(node)
        non_leaf_nodes = total_nodes - self._count_leaf_nodes(node)
        if non_leaf_nodes == 0:
            return 0.0
        return (total_nodes - 1) / non_leaf_nodes

    def _get_best_path(self, node: MCTSNode) -> List[Dict]:
        """Get the path with highest value that leads to a terminal state"""
        path = []
        current = node
        while current:
            path.append({
                'state': current.state,
                'stats': current.get_stats().to_dict()
            })
            if not current.children:
                break

            # Among children that have been visited at least once,
            # choose the one with highest success rate
            visited_children = [c for c in current.children if c.visits > 0]
            if not visited_children:
                break

            # If any child is terminal with a good score, prefer it
            terminal_children = [c for c in visited_children if c.is_terminal and c.value/max(c.visits, 1) > 0]
            if terminal_children:
                current = max(terminal_children,
                             key=lambda n: n.value / max(n.visits, 1))
            else:
                current = max(visited_children,
                         key=lambda n: n.value / max(n.visits, 1))
        return path

# Example evaluation function:
def answer_comparison_evaluator(state: str, problem: str,
                               expected_answer: float) -> Tuple[int, int]:
    """Evaluates solution by comparing extracted answers with expected answer."""
    # Extract answer from \boxed{} in the solution
    answer_match = re.search(r'oxed\{([^}]+)\}', state)
    if not answer_match:
        return (0, 1)

    solution_answer = answer_match.group(1).strip().replace("\\", "").strip()
    try:
        solution_answer = float(solution_answer)
    except ValueError:
        return (0, 1)

    return (1, 1) if np.abs(solution_answer - expected_answer) < 1e-10 else (0, 1)

In [ ]:
from openai import OpenAI
import os

# Initialize the LLM client
async_client = OpenAI(
    base_url="https://api.studio.nebius.ai/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

# Create the LLM client wrapper
llm_client = LLMClient(
    client=async_client,
    model="meta-llama/Meta-Llama-3.1-70B-Instruct",
    system_prompt=None,
)

# Initialize MCTS with the client
mcts = MCTS(
    llm_client=llm_client,
    max_depth=10,
    num_rollouts=5,
    c=0.5
)

Теперь функция `mcts.search` запустит MCTS, показывая общую статистику дерева и лучший путь решения каждые 5 итераций.

In [ ]:
# Run the search
solution, tree = mcts.search(
    problem="Inside a circle, two parallel chords are 6 units apart. One chord has length 14 and the other has length 10. Find the square of the radius of the circle.",
    evaluate_continuation=lambda state, problem: answer_comparison_evaluator(state, problem, 50.0),
    num_iterations=100,
    verbose=True
)

print(solution)


Iteration 0 - Tree Statistics:
{
  "total_nodes": 2,
  "max_depth": 1,
  "leaf_nodes": 1,
  "avg_branching": 1.0,
  "best_path": [
    {
      "state": "",
      "stats": {
        "visits": 1,
        "value": 0.2,
        "success_rate": 0.2,
        "depth": 0,
        "is_terminal": false,
        "num_children": 1
      }
    },
    {
      "state": "Let's denote the radius of the circle as r. Draw a line from the center of the circle to the midpoint of each chord, and another line from the center of the circle that is perpendicular to the chords. This will create two right triangles. Using the Pythagorean theorem, we can express the radius squared as r^2 = (7^2 + 3^2) = (5^2 + 3^2), where 7 and 5 are half the lengths of the two chords.",
      "stats": {
        "visits": 1,
        "value": 0.2,
        "success_rate": 0.2,
        "depth": 1,
        "is_terminal": false,
        "num_children": 0
      }
    }
  ]
}

Iteration 5 - Tree Statistics:
{
  "total_nodes": 7,
  "max

Если вы посмотрите на лучший путь решения, вы заметите, что `success_rate` растет с глубиной, что логично:
- на старте есть много путей, начинающихся от пня частичного решения, и многие из них неправильные.
- позже, если мы действительно находимся на пути к правильному решению, для неудач остается все меньше и меньше места.

**Вопрос к вам**. Как вы думаете, сколько звонков по LLM мы здесь сделали?

**Задание для вас**. В нашей реализации MCTS останавливается, когда исчерпывается количество итераций. Однако в более простых задачах мы можем достичь конечных узлов и прекратить создание новых раньше, поэтому критерий ранней остановки был бы удобен. Определить его можно несколькими способами, например:
* Никаких новых узлов в течение нескольких итераций,
* Сходимость оценок значений: если значения практически не меняются за ряд итераций, это может быть сигналом о том, что пора остановиться.

## Практика, часть 2: Форсирование бюджета
Хотя мы не можем позволить себе точную настройку LLM, как в [документе S1](https://arxiv.org/pdf/2501.19393), мы все равно можем применять бюджетное форсирование.
Напомним, что идея проста: заставить LLM генерировать «Подождите» (или другую фразу, заставляющую задуматься) каждый раз, когда он пытается вывести ответ — до тех пор, пока не будет достигнута целевая длина решения.
Этот подход требует определенной степени контроля над генерацией LLM, которую API не могут предложить, поэтому мы должны запустить модель локально в этом блокноте Colab. Прежде чем начать, переключитесь на компьютер с графическим процессором — графического процессора L4 будет достаточно.
Наша реализация бюджетного форсирования генерирует решение в два этапа:
1. **Фаза предварительного ожидания:**
   Пока длина решения не достигнет `wait_tokens`, мы вмешиваемся в следующую генерацию токена, используя специальный **процессор логитов**. В этом процессоре мы отслеживаем несколько маркеров и при обнаружении одного из них вставляем *последовательность ожидания* (`wait_text`) вместо последнего токена маркера. По умолчанию используемый нами LLM включает ответы в `\boxed{}`, поэтому мы используем `\boxed` в качестве маркера.
2. **Фаза после ожидания:**
   Как только длина решения достигнет `wait_tokens`, мы позволяем LLM генерироваться свободно.
Мы рекомендуем вам поэкспериментировать с увеличением бюджета для различных задач и с разными последовательностями ожидания.
**Пища для размышления: вопросы для рассмотрения**
- Что произойдет, если мы установим вероятность регистрации токена `<eos>` (конец строки) на `-inf` до достижения `wait_tokens`?
- Что произойдет, если мы подавим генерацию `\boxed{}`, установив вероятность журнала `boxed` на `-inf` после обратной косой черты (`\`)?

Прежде всего, вам необходимо загрузить токен доступа Hugging Face:

In [ ]:
!pip install -q openai

In [ ]:
import os

with open("nebius_api_key", "r") as file:
    nebius_api_key = file.read().strip()

os.environ["NEBIUS_API_KEY"] = nebius_api_key

with open("hf_access_token", "r") as file:
    hf_access_token = file.read().strip()

In [ ]:
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessorList

class BudgetForcingProcessor(torch.nn.Module):
    """Custom logits processor that forces the model to wait before generating answer markers"""
    def __init__(self, tokenizer, wait_tokens=200, debug=True, markers=None, wait_text=None):
        super().__init__()
        self.tokenizer = tokenizer
        self.wait_tokens = wait_tokens
        self.debug = debug

        # These will be set by the parent class, but provide defaults if needed
        # They're initially set here but will be overridden by BudgetForcingLLM
        self.markers = markers or {
            "boxed": {"text": "\\boxed", "case_sensitive": True},
            "eos": {"text": self.tokenizer.eos_token, "case_sensitive": True}
        }
        self.wait_text = wait_text or "\nWait, I need to think more about this..."

        # Get EOS token ID specifically
        self.eos_token_id = self.tokenizer.eos_token_id

        # Initialize token maps
        self.marker_tokens = {}
        self.wait_tokens_ids = []
        self.update_token_maps()

        # State tracking
        self.token_buffer = []
        self.intervention_active = False
        self.total_tokens_generated = 0  # Count from the beginning
        self.wait_text_inserted = False
        self.forced_wait_index = 0
        self.last_intervention_position = 0  # Track position of last intervention
        self.cooldown_tokens = 50  # Minimum tokens between interventions
        self.budget_notification_sent = False

        # Flag to indicate if the budget is exhausted
        self.budget_exhausted = False

        # Flag to signal early stopping once budget is met
        self.stop_at_budget = True

    def update_token_maps(self):
        """Update token maps when markers or wait text changes"""
        # Find how each marker is tokenized
        self.marker_tokens = {}
        for key, marker_info in self.markers.items():
            marker_text = marker_info["text"]
            if marker_text:  # Skip None values
                self.marker_tokens[key] = self.tokenizer.encode(marker_text, add_special_tokens=False)

        # Get wait text tokens
        self.wait_tokens_ids = self.tokenizer.encode(self.wait_text, add_special_tokens=False)

        # State tracking
        self.token_buffer = []
        self.intervention_active = False
        self.total_tokens_generated = 0  # Count from the beginning
        self.wait_text_inserted = False
        self.forced_wait_index = 0
        self.last_intervention_position = 0  # Track position of last intervention
        self.cooldown_tokens = 50  # Minimum tokens between interventions
        self.budget_notification_sent = False

        # Flag to indicate if the budget is exhausted
        self.budget_exhausted = False

        # Flag to signal early stopping once budget is met
        self.stop_at_budget = True

    def detect_marker_start(self, token_ids):
        """Check if the current sequence of tokens might be starting to generate any of the answer markers"""
        # For efficiency, only check recent tokens
        check_window = 10  # Check the last 10 tokens at most
        start_idx = max(0, len(token_ids) - check_window)
        recent_token_ids = token_ids[start_idx:].tolist()

        # Completely rebuild the buffer from the recent tokens to avoid stale state
        self.token_buffer = recent_token_ids

        # For debugging, show the tokens we're checking
        if self.debug:
            recent_str = self.tokenizer.decode(self.token_buffer)

        # Check if we're in cooldown period after a recent intervention
        if self.last_intervention_position > 0:
            tokens_since_intervention = len(token_ids) - self.last_intervention_position
            if tokens_since_intervention < self.cooldown_tokens:
                # Don't trigger intervention during cooldown
                return False

        # Direct check for EOS token - this still needs to be caught immediately
        if self.eos_token_id in self.token_buffer and self.total_tokens_generated < self.wait_tokens:
            if self.debug:
                print(f"EOS token detected at position {len(token_ids)}, before wait_tokens threshold!")
            self.detected_marker = "EOS token"
            return True

        # Check for markers in the current window
        token_str = self.tokenizer.decode(self.token_buffer)

        # Use the markers dictionary with its properties
        for marker_key, marker_info in self.markers.items():
            marker_text = marker_info["text"]
            case_sensitive = marker_info["case_sensitive"]

            if not marker_text or marker_key == "eos":
                continue  # Skip None or EOS markers

            # Check for the marker in the token string
            if case_sensitive:
                # Case-sensitive check
                marker_present = marker_text in token_str
            else:
                # Case-insensitive check
                marker_present = marker_text.lower() in token_str.lower()

            if marker_present:
                if self.debug:
                    print(f"{marker_key.upper()} marker detected in: '{token_str}'")
                self.detected_marker = marker_text
                return True

        return False

    def __call__(self, input_ids, scores):
        """Process logits to implement budget forcing"""
        # Count tokens from the beginning, excluding the prompt
        prompt_token_length = len(input_ids)
        self.total_tokens_generated = len(input_ids[0]) - prompt_token_length

        if self.debug and self.total_tokens_generated % 50 == 0:
            print(f"Total tokens generated: {self.total_tokens_generated}/{self.wait_tokens}")

            # Periodically show the last tokens in debug mode to help with debugging
            if len(input_ids[0]) > 10:
                recent_text = self.tokenizer.decode(input_ids[0][-10:])

        # Check if we've reached the token budget
        self.budget_exhausted = self.total_tokens_generated >= self.wait_tokens

        # If budget is exhausted and we want to stop, force EOS token
        if self.budget_exhausted and self.stop_at_budget:
            if self.debug and not self.budget_notification_sent:
                print(f"\n==== BUDGET REACHED, STOPPING FIRST STAGE ====")
                print(f"Generated {self.total_tokens_generated} tokens")
                self.budget_notification_sent = True

            # Force EOS token to stop generation
            scores[0, :] = -float('inf')
            scores[0, self.eos_token_id] = 100.0
            return scores

        # Clear the token buffer periodically to avoid keeping stale tokens
        if len(self.token_buffer) > 0 and self.total_tokens_generated % 100 == 0:
            self.token_buffer = []

        # Always block EOS until budget is exhausted to prevent early termination
        if not self.budget_exhausted:
            # Make EOS impossible - this is critical
            scores[0, self.eos_token_id] = -float('inf')

            # Reset detected_marker before checking
            self.detected_marker = None

            # Only check for specific markers, no rule-based pattern detection
            if not self.intervention_active and self.detect_marker_start(input_ids[0]):

                if self.debug:
                    print("\n==== ACTIVATING BUDGET FORCING INTERVENTION ====")
                    print(f"Detected marker: {self.detected_marker}")
                    print(f"At position {len(input_ids[0])}")
                    print(f"Current token count: {self.total_tokens_generated}/{self.wait_tokens}")

                    # Print recent context
                    if len(input_ids[0]) > 20:
                        recent_text = self.tokenizer.decode(input_ids[0][-20:])
                        print(f"Recent context: '{recent_text}'")

                # Activate intervention
                self.intervention_active = True
                self.wait_text_inserted = False
                self.forced_wait_index = 0

                # Force the start of wait text by modifying scores
                next_wait_token = self.wait_tokens_ids[0]
                scores[0, :] = -float('inf')  # Set all token probabilities to very low
                scores[0, next_wait_token] = 100.0  # Force the first wait token
                self.forced_wait_index = 1

                if self.debug:
                    print(f"Starting wait text: '{self.tokenizer.decode([next_wait_token])}'")

                return scores

        # If we're in active intervention mode, force wait text
        if self.intervention_active:
            # If we haven't started inserting wait text
            if self.forced_wait_index == 0:
                next_wait_token = self.wait_tokens_ids[0]
                scores[0, :] = -float('inf')  # Set all token probabilities to very low
                scores[0, next_wait_token] = 100.0  # Force the first wait token
                self.forced_wait_index = 1

                if self.debug:
                    print(f"Starting wait text: '{self.tokenizer.decode([next_wait_token])}'")

                return scores

            # If we're in the middle of inserting wait text
            elif not self.wait_text_inserted:
                # Check if we've inserted all wait tokens
                if self.forced_wait_index >= len(self.wait_tokens_ids):
                    self.wait_text_inserted = True
                    self.intervention_active = False  # End intervention after wait text

                    # Add a cooldown period to prevent multiple consecutive wait texts
                    self.last_intervention_position = len(input_ids[0])

                    if self.debug:
                        print("Wait text fully inserted, continuing normal generation")
                    return scores

                # Force next wait token
                next_wait_token = self.wait_tokens_ids[self.forced_wait_index]
                scores[0, :] = -float('inf')
                scores[0, next_wait_token] = 100.0
                self.forced_wait_index += 1

                return scores

        return scores


class BudgetForcingLLM:
    # Define markers at the class level for consistency
    # Each marker has: text value and case_sensitive flag
    DEFAULT_MARKERS = {
        "correct_answer": {"text": """the correct answer is:""", "case_sensitive": False},
        "final_answer": {"text": """the final answer is:""", "case_sensitive": False},
        "boxed": {"text": "\\boxed", "case_sensitive": True},
        "eos": {"text": None, "case_sensitive": True}  # Will be filled with the tokenizer's EOS token
    }
    DEFAULT_WAIT_TEXT = "\nWait, I need to think more about this..."

    def __init__(
        self,
        model_id: str = "meta-llama/Llama-3.1-8B",
        device: str = "cuda",
        wait_tokens: int = 200,  # Number of tokens to wait before allowing markers
        debug: bool = True,
        hf_access_token: str = None,
        markers: dict = None,
        wait_text: str = None
    ):
        """
        Initialize the Budget Forcing implementation for Llama models.

        Args:
            model_id: HuggingFace model ID
            device: Device to run the model on (cuda/cpu)
            wait_tokens: Number of tokens to generate before allowing answer markers
            debug: Whether to print debug information
            markers: Custom markers dictionary (optional)
            wait_text: Custom wait text (optional)
        """
        print(f"Loading model {model_id} on {device}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_access_token)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True,
            token=hf_access_token
        )
        self.model.to(device)
        self.device = device
        self.wait_tokens = wait_tokens
        self.debug = debug

        # Initialize markers and wait text
        self.markers = markers or self.DEFAULT_MARKERS.copy()
        # Ensure EOS token is correctly set
        if "eos" in self.markers:
            self.markers["eos"]["text"] = self.tokenizer.eos_token
        self.wait_text = wait_text or self.DEFAULT_WAIT_TEXT

        # Create our logits processor with our markers and wait text
        self.budget_processor = BudgetForcingProcessor(
            self.tokenizer,
            wait_tokens=wait_tokens,
            debug=debug,
            markers=self.markers,
            wait_text=self.wait_text
        )

    def clean_markers(self, text):
        """Remove only markers from generated text, keeping wait text intact"""
        cleaned_text = text

        # Remove all markers from the text using the dictionary with its properties
        for marker_key, marker_info in self.budget_processor.markers.items():
            marker_text = marker_info["text"]
            case_sensitive = marker_info["case_sensitive"]

            if marker_text and marker_key != "eos":  # Skip None values and EOS token
                pattern = re.escape(marker_text)

                if case_sensitive:
                    # Case-sensitive removal
                    cleaned_text = re.sub(pattern, "", cleaned_text)
                else:
                    # Case-insensitive removal
                    cleaned_text = re.sub(pattern, "", cleaned_text, flags=re.IGNORECASE)

        return cleaned_text

    def generate(
        self,
        prompt: str,
        max_new_tokens: int = 512,
        temperature: float = 0.7,
        top_p: float = 0.9
    ) -> str:
        """Generate text with two-stage budget forcing"""
        if self.debug:
            print("\n===== STARTING TWO-STAGE GENERATION =====")
            print(f"Wait tokens: {self.wait_tokens}")
            print(f"Max new tokens: {max_new_tokens}")

        # Customize prompt to strongly encourage the model to use the marker


        # STAGE 1: Generate with budget forcing until budget is exhausted
        # -------------------------------------------------------------------
        # Reset the processor state completely
        self.budget_processor = BudgetForcingProcessor(
            self.tokenizer,
            wait_tokens=self.wait_tokens,
            debug=self.debug
        )

        # Enable "stop at budget" mode
        self.budget_processor.stop_at_budget = True

        # Store the prompt token length for accurate token counting
        self.budget_processor.prompt_token_length = len(self.tokenizer.encode(prompt, add_special_tokens=False))

        # Tokenize input
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)

        # Define our logits processor list
        logits_processor = LogitsProcessorList([self.budget_processor])

        # Generate first stage with our custom processor
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=min(self.wait_tokens + 100, max_new_tokens),  # Just enough to reach budget
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                logits_processor=logits_processor,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        # Decode the output and extract only the generated text
        full_output = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
        first_stage_text = full_output[len(prompt):]

        if self.debug:
            print(f"\nStage 1 generated {len(self.tokenizer.encode(first_stage_text))} tokens")

        # Clean the generated text to remove markers but keep wait text
        cleaned_text = self.clean_markers(first_stage_text)

        if self.debug:
            print(f"Cleaned text for stage 2: {cleaned_text}")

        # STAGE 2: Continue generation from cleaned text without budget forcing
        # -------------------------------------------------------------------
        # Create new prompt with the cleaned text from stage 1
        stage2_prompt = prompt + cleaned_text

        # Calculate remaining tokens for stage 2
        tokens_used_stage1 = len(self.tokenizer.encode(cleaned_text))
        remaining_tokens = max_new_tokens - tokens_used_stage1

        if remaining_tokens <= 0:
            if self.debug:
                print("No tokens remaining for stage 2, returning stage 1 result")
            return cleaned_text

        if self.debug:
            print(f"\n===== STARTING SECOND STAGE GENERATION =====")
            print(f"Tokens remaining: {remaining_tokens}")

        # Tokenize the new prompt
        stage2_inputs = self.tokenizer(stage2_prompt, return_tensors="pt").to(self.device)

        # Generate without budget forcing for the second stage
        with torch.no_grad():
            stage2_output_ids = self.model.generate(
                **stage2_inputs,
                max_new_tokens=remaining_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        # Extract only the new generated text (beyond the stage 2 prompt)
        stage2_full_output = self.tokenizer.decode(stage2_output_ids[0], skip_special_tokens=True)
        stage2_generated_text = stage2_full_output[len(stage2_prompt):]

        if self.debug:
            print(f"Stage 2 generated {len(self.tokenizer.encode(stage2_generated_text))} tokens")

        # Combine the cleaned first stage and second stage text
        final_output = cleaned_text + stage2_generated_text

        if self.debug:
            print(f"\nTotal tokens generated: {len(self.tokenizer.encode(final_output))}/{max_new_tokens}")

        return final_output

Давайте создадим бюджетный курс LLM. Мы выбрали небольшой LLM, чтобы быть уверенными, что не истощим графический процессор даже при работе с длинными решениями.

In [ ]:
# Example usage
budget_llm = BudgetForcingLLM(
        model_id="Qwen/Qwen2.5-1.5B-Instruct",
        wait_tokens=2048,
        debug=True,
        hf_access_token=hf_access_token
    )

Loading model Qwen/Qwen2.5-1.5B-Instruct on cuda


Давайте попробуем генерацию!

In [ ]:
# Problem 63 from the AIME dataset: https://huggingface.co/datasets/AI-MO/aimo-validation-aime
math_problem = """Let\n$$x^8 + 3x^4 - 4 = p_1(x) p_2(x) \\dotsm p_k(x),$$\nwhere each non-constant polynomial $p_i(x)$ is monic with integer coefficients, and cannot be factored further over the integers.  Compute $p_1(1) + p_2(1) + \\dots + p_k(1).$"""

    # Generate with budget forcing
print("\n--- Generating solution with budget forcing ---")
output = budget_llm.generate(
    math_problem,
    max_new_tokens=3096,
    temperature=0.8
)

print("\nGenerated solution:")
print(output)




--- Generating solution with budget forcing ---

===== STARTING TWO-STAGE GENERATION =====
Wait tokens: 2048
Max new tokens: 3096
Total tokens generated: 100/2048
Total tokens generated: 150/2048
Total tokens generated: 200/2048
Total tokens generated: 250/2048
Total tokens generated: 300/2048
Total tokens generated: 350/2048
Total tokens generated: 400/2048
Total tokens generated: 450/2048
Total tokens generated: 500/2048
Total tokens generated: 550/2048
Total tokens generated: 600/2048
Total tokens generated: 650/2048
Total tokens generated: 700/2048
Total tokens generated: 750/2048
Total tokens generated: 800/2048
Total tokens generated: 850/2048
BOXED marker detected in: ']
Therefore, the answer is:
\[ \boxed'

==== ACTIVATING BUDGET FORCING INTERVENTION ====
Detected marker: \boxed
At position 863
Current token count: 862/2048
Recent context: ' 2 + 0 = 8. \]
Therefore, the answer is:
\[ \boxed'
Starting wait text: '
'
Wait text fully inserted, continuing normal generation
Total t

In [ ]:
math_problem = """Inside a circle, two parallel chords are 6 units apart. One chord has length 14 and the other has length 10. Find the radius of the circle."""

    # Generate with budget forcing
print("\n--- Generating solution with budget forcing ---")
output = budget_llm.generate(
    math_problem,
    max_new_tokens=3096,
    temperature=0.8
)

print("\nGenerated solution:")
print(output)




--- Generating solution with budget forcing ---

===== STARTING TWO-STAGE GENERATION =====
Wait tokens: 2048
Max new tokens: 3096
Total tokens generated: 50/2048
Total tokens generated: 100/2048
Total tokens generated: 150/2048
Total tokens generated: 200/2048
Total tokens generated: 250/2048
Total tokens generated: 300/2048
Total tokens generated: 350/2048
Total tokens generated: 400/2048
Total tokens generated: 450/2048
Total tokens generated: 500/2048
Total tokens generated: 550/2048
Total tokens generated: 600/2048
Total tokens generated: 650/2048
BOXED marker detected in: ', the radius of the circle is \(\boxed'

==== ACTIVATING BUDGET FORCING INTERVENTION ====
Detected marker: \boxed
At position 694
Current token count: 693/2048
Recent context: ' 5\sqrt{2} \]

So, the radius of the circle is \(\boxed'
Starting wait text: '
'
Total tokens generated: 700/2048
Wait text fully inserted, continuing normal generation
Total tokens generated: 750/2048
Total tokens generated: 800/2048
To

Если вы поэкспериментируете с нашей реализацией бюджетного форсирования, вы заметите, что **Контекстное обучение** становится неожиданной проблемой для этой модели. Действительно, после нескольких принуждений «Подождите» после того, что приводит к ответу, модель становится тревожной, и она часто демонстрирует такое же «ожидающее» поведение даже после того, как вмешательства закончились. Так что, действительно, дополнительная тонкая настройка этой модели не помешала бы.
**Ваша задача**. Поэкспериментируйте с различными проблемами и продолжительностью окна вмешательства. Попробуйте разные температуры (мы не зря сделали ее относительно высокой). Если ваш графический процессор позволяет, попробуйте модель большего размера. Сможете ли вы преодолеть тревогу модели без тонкой настройки?